# Pandas Notes: Merge and Concatenate DataFrames

## Learning objectives
- Understand when to use `merge()` versus `concat()`.
- Join DataFrames using a common column or their indexes.
- Choose between `inner`, `left`, `right`, and `outer` joins.
- Concatenate data vertically or horizontally.
- Understand `join`, `keys`, `ignore_index`, `suffixes`, and missing values.

## 1. Import pandas

`pandas` is a Python library used to work with tabular data. The common convention is to import it as `pd`.

- `import pandas as pd`: loads pandas and gives it the short alias `pd`.
- `pd.DataFrame(...)`: creates a table with rows and columns.
- `pd.Series(...)`: creates a one-dimensional labeled sequence of values.

In [20]:
import pandas as pd

## 2. `pd.merge()` — combine rows using matching values

Use `pd.merge()` when two tables are related through a common key, such as `customer_id`, `product_id`, or `A`.

### Example: merge on a common column

The `A` column appears in both DataFrames. By default, `pd.merge()` performs an **inner join**, so only values of `A` found in both tables are returned.

**Syntax:** `pd.merge(left, right, on=None, how="inner", ...)`

- `left`: the first DataFrame.
- `right`: the second DataFrame.
- `on`: the column name used as the matching key. The column must exist in both DataFrames.
- `how`: specifies which matching/non-matching keys to keep. Its default is `"inner"`.

In [2]:
left = pd.DataFrame({
    "A": [1, 2, 3, 4],
    "B": [11, 12, 13, 14]
})

right = pd.DataFrame({
    "A": [1, 2, 3, 5],
    "C": [21, 22, 23, 24]
})

pd.merge(left, right, on="A")

,A,B,C
0,1,11,21
1,2,12,22
2,3,13,23


**What happens?** Keys `1`, `2`, and `3` occur in both tables, so those rows appear. Key `4` exists only on the left and key `5` only on the right, so the default inner join excludes them.

## 3. The `how` parameter: four common join types

Use the same two DataFrames to compare join behavior:

- **`how="inner"`**: keep only keys found in both DataFrames.
- **`how="left"`**: keep every key from the left DataFrame; unmatched right-side values become `NaN`.
- **`how="right"`**: keep every key from the right DataFrame; unmatched left-side values become `NaN`.
- **`how="outer"`**: keep every key from either DataFrame; unmatched values become `NaN`.

`NaN` represents a missing value in the result.

In [3]:
inner_result = pd.merge(left, right, on="A", how="inner")
left_result = pd.merge(left, right, on="A", how="left")
right_result = pd.merge(left, right, on="A", how="right")
outer_result = pd.merge(left, right, on="A", how="outer")

print("INNER JOIN")
display(inner_result)

print("LEFT JOIN")
display(left_result)

print("RIGHT JOIN")
display(right_result)

print("OUTER JOIN")
display(outer_result)

INNER JOIN


,A,B,C
0,1,11,21
1,2,12,22
2,3,13,23


LEFT JOIN


,A,B,C
0,1,11,21.0
1,2,12,22.0
2,3,13,23.0
3,4,14,NaN


RIGHT JOIN


,A,B,C
0,1,11.0,21
1,2,12.0,22
2,3,13.0,23
3,5,NaN,24


OUTER JOIN


,A,B,C
0,1,11.0,21.0
1,2,12.0,22.0
2,3,13.0,23.0
3,4,14.0,NaN
4,5,NaN,24.0


## 4. `indicator=True` — see where each row came from

`indicator=True` adds a column named `_merge` to the result. It labels each key as:
- `left_only`: key exists only in the left DataFrame.
- `right_only`: key exists only in the right DataFrame.
- `both`: key exists in both DataFrames.

This is useful for checking data quality and finding records that do not match.

In [5]:
pd.merge(left, right, on="A", how="outer", indicator=True)

,A,B,C,_merge
0,1,11.0,21.0,both
1,2,12.0,22.0,both
2,3,13.0,23.0,both
3,4,14.0,NaN,left_only
4,5,NaN,24.0,right_only


## 5. `suffixes` — resolve repeated column names

If both DataFrames contain columns with the same name besides the join key, pandas adds suffixes to distinguish them.

- `suffixes=("_left", "_right")`: labels overlapping columns from the left and right DataFrames.
- The value must be a pair (a tuple or similar two-item sequence).

In this example, both tables contain `B`, so the result uses `B_left` and `B_right`.

In [6]:
left_scores = pd.DataFrame({"id": [1, 2, 3], "B": [70, 80, 90]})
right_scores = pd.DataFrame({"id": [1, 2, 4], "B": [75, 82, 88]})

pd.merge(
    left_scores,
    right_scores,
    on="id",
    how="outer",
    suffixes=("_left", "_right")
)

,id,B_left,B_right
0,1,70.0,75.0
1,2,80.0,82.0
2,3,90.0,NaN
3,4,NaN,88.0


## 6. Merge using indexes

When the matching values are in the **index** rather than a regular column, use `left_index=True` and `right_index=True`.

- `left_index=True`: use the left DataFrame's index as a key.
- `right_index=True`: use the right DataFrame's index as a key.

This is different from `on="A"`, which matches using a column.

In [7]:
left_indexed = pd.DataFrame({"name": ["Asha", "Ravi", "Neha"]}, index=[101, 102, 103])
right_indexed = pd.DataFrame({"score": [85, 91, 78]}, index=[101, 102, 104])

pd.merge(
    left_indexed,
    right_indexed,
    left_index=True,
    right_index=True,
    how="outer"
)

,name,score
101,Asha,85.0
102,Ravi,91.0
103,Neha,NaN
104,NaN,78.0


## 7. `pd.concat()` — stack or place objects side by side

Use `pd.concat()` to combine multiple Series or DataFrames along an axis.

**Syntax:** `pd.concat(objs, axis=0, join="outer", ignore_index=False, keys=None)`

- `objs`: a list (or other iterable) of Series/DataFrames to combine.
- `axis=0`: combine vertically, adding rows. This is the default.
- `axis=1`: combine horizontally, adding columns.
- `join="outer"`: keep the union of the other axis labels (default).
- `join="inner"`: keep only the labels shared by all objects on the other axis.
- `ignore_index=True`: create a fresh sequential index on the concatenation axis.
- `keys`: add an outer index label to identify each input object.

Unlike `merge()`, `concat()` usually aligns objects by their index and column labels; it does not match rows using a chosen key column.

### Example A: concatenate Series vertically

`pd.concat([sr1, sr2])` puts the values from the second Series after the first. By default, the original index labels are retained, so duplicate index labels can occur.

In [8]:
sr1 = pd.Series([1, 2, 3, 4], name="first")
sr2 = pd.Series([21, 31, 41, 51], name="second")

pd.concat([sr1, sr2])

0     1
1     2
2     3
3     4
0    21
1    31
2    41
3    51
dtype: int64

To reset the combined Series to a continuous index, set `ignore_index=True`.

In [9]:
pd.concat([sr1, sr2], ignore_index=True)

0     1
1     2
2     3
3     4
4    21
5    31
6    41
7    51
dtype: int64

### Example B: concatenate DataFrames vertically (`axis=0`)

Rows from `d2` are added below rows from `d1`. Because the two DataFrames have the same column names, their values align into the same columns.

In [10]:
d1 = pd.DataFrame({"A": [1, 2, 3], "B": [11, 12, 13]})
d2 = pd.DataFrame({"A": [4, 5], "B": [21, 22]})

pd.concat([d1, d2], axis=0)

,A,B
0,1,11
1,2,12
2,3,13
0,4,21
1,5,22


### Example C: concatenate DataFrames horizontally (`axis=1`)

Columns from the second DataFrame are placed beside columns from the first. pandas aligns rows by index label, not simply by their visual position.

In [11]:
d3 = pd.DataFrame({"A": [1, 2, 3]}, index=[0, 1, 2])
d4 = pd.DataFrame({"B": [21, 22, 23]}, index=[0, 1, 2])

pd.concat([d3, d4], axis=1)

,A,B
0,1,21
1,2,22
2,3,23


## 8. `join="outer"` versus `join="inner"` in `concat()`

For `concat()`, `join` controls how labels on the **other axis** are aligned.

- With `axis=0`, the other axis is the columns. `outer` keeps all columns; `inner` keeps only columns shared by all DataFrames.
- With `axis=1`, the other axis is the row index. `outer` keeps all index labels; `inner` keeps only index labels shared by all DataFrames.

The following example uses `axis=1` and different index lengths.

In [12]:
short_df = pd.DataFrame({"A": [1, 2, 3, 4]}, index=[0, 1, 2, 3])
small_df = pd.DataFrame({"B": [21, 22]}, index=[0, 1])

print("OUTER: all index labels are retained")
display(pd.concat([short_df, small_df], axis=1, join="outer"))

print("INNER: only shared index labels are retained")
display(pd.concat([short_df, small_df], axis=1, join="inner"))

OUTER: all index labels are retained


,A,B
0,1,21.0
1,2,22.0
2,3,NaN
3,4,NaN


INNER: only shared index labels are retained


,A,B
0,1,21
1,2,22


## 9. `keys` — identify the source of concatenated data

`keys` adds an outer level to the result's index. This is useful when you want to remember which DataFrame each row came from.

The number and order of keys should correspond to the objects being concatenated.

In [13]:
sales_jan = pd.DataFrame({"sales": [100, 150]}, index=["store_1", "store_2"])
sales_feb = pd.DataFrame({"sales": [120, 170]}, index=["store_1", "store_2"])

pd.concat([sales_jan, sales_feb], keys=["January", "February"])

sales
January  store_1    100
         store_2    150
February store_1    120
         store_2    170

## 10. Concatenating DataFrames with different columns

When stacking rows (`axis=0`), pandas aligns columns by name. If one DataFrame does not have a column that another has, the missing cells are filled with `NaN` under the default `join="outer"`.

In [14]:
only_d = pd.DataFrame({"D": [1, 2, 3]})
a_b = pd.DataFrame({"A": [10, 20, 30], "B": [21, 22, 23]})

pd.concat([only_d, a_b], axis=0)

,D,A,B
0,1.0,NaN,NaN
1,2.0,NaN,NaN
2,3.0,NaN,NaN
0,NaN,10.0,21.0
1,NaN,20.0,22.0
2,NaN,30.0,23.0


## 11. `merge()` versus `concat()` — quick comparison

| Feature | `pd.merge()` | `pd.concat()` |
|---|---|---|
| Main purpose | Match related records using key columns or indexes | Stack objects or place them side by side |
| Typical use | Join customers to orders using `customer_id` | Append monthly data files with the same columns |
| Matching behavior | Uses specified key(s) | Aligns by index and column labels |
| Common options | `on`, `how`, `left_index`, `right_index`, `suffixes`, `indicator` | `axis`, `join`, `ignore_index`, `keys` |

**Rule of thumb:** Use `merge()` when you need to match records by a key. Use `concat()` when you need to stack tables or attach them along an axis.

## 12. Common mistakes to avoid

1. **Using `=` instead of a comparison in a condition:** use `isna()` / `notna()` for missing values in pandas, rather than comparing to `None` or `NaN` directly.
2. **Forgetting the default join:** `merge()` defaults to `how="inner"`; `concat()` defaults to `axis=0` and `join="outer"`.
3. **Assuming `axis=1` matches a key column:** horizontal `concat()` aligns on index labels. If you need to match on an ID column, use `merge()`.
4. **Overlapping column names:** use `suffixes` in `merge()` to make the origin of each column clear.
5. **Duplicate index labels after concatenation:** use `ignore_index=True` when a fresh sequential index is appropriate.

## 13. Practice exercises

Try these yourself before checking the suggested solutions.

1. Merge two DataFrames on `employee_id` using a left join.
2. Use `indicator=True` with an outer merge and identify rows that exist only on one side.
3. Concatenate two DataFrames vertically and reset the index.
4. Concatenate two DataFrames horizontally using `join="inner"`.
5. Concatenate two monthly DataFrames with `keys=["January", "February"]`.

### Suggested solutions

In [15]:
# Exercise 1: left join
employees = pd.DataFrame({"employee_id": [1, 2, 3], "name": ["Asha", "Ravi", "Neha"]})
bonuses = pd.DataFrame({"employee_id": [1, 3], "bonus": [500, 800]})

pd.merge(employees, bonuses, on="employee_id", how="left")

,employee_id,name,bonus
0,1,Asha,500.0
1,2,Ravi,NaN
2,3,Neha,800.0


In [16]:
# Exercise 2: find whether keys matched
pd.merge(employees, bonuses, on="employee_id", how="outer", indicator=True)

,employee_id,name,bonus,_merge
0,1,Asha,500.0,both
1,2,Ravi,NaN,left_only
2,3,Neha,800.0,both


In [17]:
# Exercise 3: stack and reset index
pd.concat([d1, d2], axis=0, ignore_index=True)

,A,B
0,1,11
1,2,12
2,3,13
3,4,21
4,5,22


In [18]:
# Exercise 4: side-by-side concatenation with shared index labels only
pd.concat([short_df, small_df], axis=1, join="inner")

,A,B
0,1,21
1,2,22


In [19]:
# Exercise 5: label the source DataFrame
pd.concat([sales_jan, sales_feb], keys=["January", "February"])

sales
January  store_1    100
         store_2    150
February store_1    120
         store_2    170

## Final recap

- `pd.merge()` joins records using a common column or index.
- `how` controls which keys are kept: `inner`, `left`, `right`, or `outer`.
- `indicator=True` shows whether each key came from the left, right, or both.
- `suffixes` distinguishes overlapping non-key columns.
- `pd.concat()` combines Series/DataFrames vertically (`axis=0`) or horizontally (`axis=1`).
- `join`, `ignore_index`, and `keys` control label alignment, index resetting, and source labeling.